In [1]:
import requests
import datetime
import pandas as pd

In [2]:
# API Key
API_KEY = 'ksnxCfuMd8YcYtNVDrqZKBY0aZeDMLX8'

In [3]:
def get_sd(ticker):
    """
    Fetches standard deviation for a given stock over 1Y, 5Y, and 10Y.
    Returns a dictionary for easy combination with CAGR data.
    """
    base_url = 'https://financialmodelingprep.com/api/v3/technical_indicator/1day/'

    # Define standard deviation lookback periods
    periods = {"1Y": 252, "5Y": 1260, "10Y": 2520}
    sd_values = {}

    for key, period in periods.items():
        url = f"{base_url}{ticker}?type=standardDeviation&period={period}&apikey={API_KEY}"
        response = requests.get(url)
        data = response.json()

        if isinstance(data, list) and len(data) > 0:
            # Extract latest standard deviation value
            sd_values[key] = round(data[0].get('standardDeviation', None), 4)  # 4 decimal places
        else:
            sd_values[key] = None  # Assign None if data is unavailable

    return sd_values  # Returns {'1Y': value, '5Y': value, '10Y': value}

In [4]:
def get_cagr(ticker):
    """
    Fetches CAGR for a given stock over 1Y, 5Y, and 10Y.
    Returns a dictionary for easy combination with SD data.
    """
    base_url = 'https://financialmodelingprep.com/api/v3/historical-price-full/'
    years_list = [1, 5, 10]
    cagr_values = {}

    for years in years_list:
        # Calculate date range
        end_date = datetime.date.today()
        start_date = end_date - datetime.timedelta(days=years * 365)

        # API Request
        url = f"{base_url}{ticker}?from={start_date}&to={end_date}&apikey={API_KEY}"
        response = requests.get(url)
        data = response.json()

        if "historical" in data and len(data["historical"]) > 0:
            # Sort data by date (oldest to newest)
            historical_data = sorted(data["historical"], key=lambda x: x["date"])
            P_start = historical_data[0]["close"]
            P_end = historical_data[-1]["close"]

            # Calculate CAGR
            cagr = ((P_end / P_start) ** (1 / years)) - 1
            cagr_values[f"{years}Y"] = round(cagr * 100, 2)  # Convert to percentage
        else:
            cagr_values[f"{years}Y"] = None  # Assign None if data is unavailable

    return cagr_values  # Returns {'1Y': value, '5Y': value, '10Y': value}


In [5]:
def get_stock_data(ticker):
    """
    Combines Standard Deviation and CAGR data into a single dictionary per stock.
    """
    sd_data = get_sd(ticker)
    cagr_data = get_cagr(ticker)

    return {
        "Ticker": ticker,
        "1Y SD": sd_data["1Y"],
        "5Y SD": sd_data["5Y"],
        "10Y SD": sd_data["10Y"],
        "1Y CAGR": cagr_data["1Y"],
        "5Y CAGR": cagr_data["5Y"],
        "10Y CAGR": cagr_data["10Y"]
    }

In [6]:
def get_multiple_stocks_data(tickers):
    """
    Fetches SD and CAGR for multiple stocks and returns a structured DataFrame.
    """
    stock_data_list = [get_stock_data(ticker) for ticker in tickers]
    df = pd.DataFrame(stock_data_list)
    return df

In [7]:
tickers = ["AAPL", "MSFT", "GOOGL"]
df = get_multiple_stocks_data(tickers)
print(df)

  Ticker    1Y SD    5Y SD    10Y SD  1Y CAGR  5Y CAGR  10Y CAGR
0   AAPL  24.8005  41.5446   67.2497    25.26    25.57     21.39
1   MSFT  16.0986  79.5156  125.0268    -7.71    19.27     24.88
2  GOOGL  13.9491  33.5735   45.2777    20.66    22.45     19.53
